In [19]:
# Assignment 12
# SECTION 1: Data Mirroring (Augmentation)

import pandas as pd
import os
import glob

folders = [
    "../../data/kinect_good_preprocessed_not_cut_A11_mediapipe",
    "../../data/kinect_good_preprocessed_A9_mediapipe"
]

pairs = [
    ('left_shoulder', 'right_shoulder'),
    ('left_elbow', 'right_elbow'),
    ('left_wrist', 'right_wrist'),
    ('left_hip', 'right_hip'),
    ('left_knee', 'right_knee'),
    ('left_ankle', 'right_ankle')
]

for target_folder in folders:
    all_files = glob.glob(os.path.join(target_folder, "*.csv"))
    print(f"Processing folder: {target_folder} ({len(all_files)} files found)")

    count = 0
    for f in all_files:
        if "_mirrored" in f:
            continue

        df = pd.read_csv(f)
        mirrored_df = df.copy()

        x_cols = [c for c in df.columns if "_3d_x" in c]
        mirrored_df[x_cols] = 1.0 - df[x_cols]

        for left, right in pairs:
            for axis in ['x', 'y', 'z']:
                l_col = f"{left}_3d_{axis}"
                r_col = f"{right}_3d_{axis}"

                if l_col in df.columns and r_col in df.columns:
                    mirrored_df[l_col] = df[r_col]
                    mirrored_df[r_col] = df[l_col]

                    if axis == 'x':
                        mirrored_df[l_col] = 1.0 - mirrored_df[l_col]
                        mirrored_df[r_col] = 1.0 - mirrored_df[r_col]

        new_path = f.replace(".csv", "_mirrored.csv")
        mirrored_df.to_csv(new_path, index=False)
        count += 1

    print(f"Successfully created {count} mirrored files in {target_folder}.\n")

print("All folders processed.")



Processing folder: ../../data/kinect_good_preprocessed_not_cut_A11_mediapipe (360 files found)


KeyboardInterrupt: 

In [ ]:

import pandas as pd
import os
import glob
from tqdm import tqdm

cut_folder   = "../../data/kinect_good_preprocessed_A9_mediapipe"
uncut_folder = "../../data/kinect_good_preprocessed_not_cut_A11_mediapipe"
output_folder = "../../data/labeled_files"

os.makedirs(output_folder, exist_ok=True)

uncut_files = glob.glob(os.path.join(uncut_folder, "*.csv"))
print(f"Found {len(uncut_files)} uncut files. Starting labeling...")

for uncut_path in tqdm(uncut_files):
    filename = os.path.basename(uncut_path)
    cut_path = os.path.join(cut_folder, filename)

    if not os.path.exists(cut_path):
        print(f"Warning: No cut file found for {filename}. Skipping.")
        continue

    df_uncut = pd.read_csv(uncut_path)
    df_cut   = pd.read_csv(cut_path)

    start_frame = df_cut['FrameNo'].min()
    end_frame   = df_cut['FrameNo'].max()

    df_uncut['activity_label'] = (
        (df_uncut['FrameNo'] >= start_frame) &
        (df_uncut['FrameNo'] <= end_frame)
    ).astype(int)

    save_path = os.path.join(output_folder, filename)
    df_uncut.to_csv(save_path, index=False)

print(f"\nDone! Labeled files saved to: {output_folder}")



Found 360 uncut files. Starting labeling...


 36%|███▌      | 129/360 [00:01<00:02, 106.84it/s]

 98%|█████████▊| 353/360 [00:03<00:00, 116.05it/s]

100%|██████████| 360/360 [00:03<00:00, 108.68it/s]


Done! Labeled files saved to: ../../data/labeled_files


In [ ]:

def get_feat_cols(columns):
    """Return only the 3D joint coordinate columns."""
    return [c for c in columns if '_3d_' in c]


ignore_list = [
    "B2_kinect.csv", "B3_kinect.csv", "B4_kinect.csv",
    "B5_kinect.csv", "A130_kinect.csv"
]

experiment_variants = [
    # 2. Dense Uni-directional LSTM (Testing if look-ahead is necessary)
    {
        "note": "LSTM_Dense_Uni",
        "seed": 42,
        "seq_length": 5,
        "hidden_size": 128,
        "lr": 0.0005,
        "batch_size": 64,
        "epochs": 30,
        "patience": 5,
        "num_layers": 2,
        "dropout": 0.1,
        "input_size": 39,
        "use_scaling": False,
        "activation": "identity",
        "init": "default",
        "bidirectional": False,
        "rnn_type": "LSTM",
    },
    # 3. Lightweight Responsive GRU (Minimalist architecture)
    {
        "note": "GRU_Light_Rapid",
        "seed": 42,
        "seq_length": 5,
        "hidden_size": 64,
        "lr": 0.001,
        "batch_size": 128,
        "epochs": 30,
        "patience": 5,
        "num_layers": 1,
        "dropout": 0.0,
        "input_size": 39,
        "use_scaling": True,
        "activation": "relu",
        "init": "kaiming",
        "bidirectional": True,
        "rnn_type": "GRU",
    },
        {
        "note": "LSTM",
        "seed": 42,
        "seq_length": 5,
        "hidden_size": 64,
        "lr": 0.0005,
        "batch_size": 128,
        "epochs": 30,
        "patience": 5,
        "num_layers": 1,
        "dropout": 0.0,
        "input_size": 39,
        "use_scaling": True,
        "activation": "relu",
        "init": "kaiming",
        "bidirectional": True,
        "rnn_type": "LSTM",
    },
]



In [ ]:

import torch
import torch.nn as nn
from torch.utils.data import Dataset
import torch.nn.init as init
import random
import numpy as np



class ActivityGatekeeper(nn.Module):
    def __init__(self, config):
        super(ActivityGatekeeper, self).__init__()

        self.bidirectional = config.get("bidirectional", False)
        multiplier = 2 if self.bidirectional else 1

        rnn_type = config.get("rnn_type", "LSTM")

        if rnn_type == "GRU":
            self.rnn = nn.GRU(
                config["input_size"],
                config["hidden_size"],
                config["num_layers"],
                batch_first=True,
                dropout=config["dropout"] if config["num_layers"] > 1 else 0,
                bidirectional=self.bidirectional,
            )
        else:
            self.rnn = nn.LSTM(
                config["input_size"],
                config["hidden_size"],
                config["num_layers"],
                batch_first=True,
                dropout=config["dropout"] if config["num_layers"] > 1 else 0,
                bidirectional=self.bidirectional,
            )

        if config.get("activation") == "relu":
            self.act = nn.ReLU()
        elif config.get("activation") == "tanh":
            self.act = nn.Tanh()
        else:
            self.act = nn.Identity()

        self.fc = nn.Linear(config["hidden_size"] * multiplier, 1)

        if config.get("init") == "xavier":
            init.xavier_uniform_(self.fc.weight)
        elif config.get("init") == "kaiming":
            init.kaiming_normal_(self.fc.weight)

    def forward(self, x):
        out, _ = self.rnn(x)
        last_step = out[:, -1, :]
        activated = self.act(last_step)
        return self.fc(activated)


class ActivityDataset(Dataset):
    def __init__(self, file_paths, config, scaler=None):
        self.X, self.y = [], []
        seq_length = config["seq_length"]
        for path in file_paths:
            df = pd.read_csv(path)
            # FIX: Use unified get_feat_cols — identical filter to scaler fit and inference
            feat_cols = get_feat_cols(df.columns)
            X_data = df[feat_cols].values.astype('float32')
            y_data = df['activity_label'].values.astype('float32')
            if scaler is not None:
                X_data = scaler.transform(X_data)
            if len(df) < seq_length:
                continue
            for i in range(len(df) - seq_length + 1):
                self.X.append(X_data[i: i + seq_length])
                self.y.append(y_data[i + seq_length - 1])

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx]), torch.tensor([self.y[idx]])


def get_with_mirrors(file_list):
    res = []
    for f in file_list:
        res.append(f)
        m = f.replace(".csv", "_mirrored.csv")
        if os.path.exists(m):
            res.append(m)
    return res





def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"✅ Random seed set to: {seed}")

def compute_pos_weight(file_list, device):
    """Compute BCE pos_weight from a list of CSV files."""
    labels = []
    for f in file_list:
        labels.extend(pd.read_csv(f)['activity_label'].values)
    neg, pos = np.bincount(np.array(labels).astype(int))
    return torch.tensor([neg / pos], dtype=torch.float).to(device)



In [ ]:
import mlflow
import torch.optim as optim
from torch.utils.data import DataLoader
import joblib
import dagshub
import sys
sys.path.append("../../scripts")
import ml_utils as mlutils
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler

dagshub.init(repo_owner="SamuelFredricBerg", repo_name="4dt907", mlflow=True)
project_name = "Start_Stop_Predictor_ModelV2"
utils = mlutils.mlutils(project_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_path = "../../data/labeled_files/"
originals = sorted([
    f for f in glob.glob(os.path.join(data_path, "*.csv"))
    if "_mirrored" not in f and os.path.basename(f) not in ignore_list
])

kf = KFold(n_splits=10, shuffle=True, random_state=42)

for config in experiment_variants:
    set_seed(config["seed"])

    best_config_for_run = config.copy()
    best_overall_mae    = float('inf')

    # FIX: Per-variant checkpoint path prevents stale weights from a previous
    # variant being loaded when architectures differ between runs.
    ckpt_path  = f"gatekeeper_best_{config['note']}.pth"
    scaler_path = f"scaler_best_{config['note']}.joblib"

    # Remove any leftover file from a previous interrupted run
    for p in (ckpt_path, scaler_path):
        if os.path.exists(p):
            os.remove(p)

    with mlflow.start_run(run_name="A12 - new metrics") as run:
        mlflow.log_params(config)

        all_start_diffs = []
        all_stop_diffs  = []
        all_file_results = []

        for fold, (train_idx, test_idx) in enumerate(kf.split(originals)):
            print(f"\n--- Fold {fold + 1} ---")
            train_orig = [originals[i] for i in train_idx]
            test_orig  = [originals[i] for i in test_idx]

            # FIX: Split train into train/val BEFORE fitting the scaler so the
            # scaler never sees validation data (avoids leaking value ranges
            # into early-stopping decisions).
            random.shuffle(train_orig)  # FIX: Shuffle to avoid positional bias in val split
            split_idx   = int(len(train_orig) * 0.9)
            train_slice = train_orig[:split_idx]
            val_slice   = train_orig[split_idx:]

            # FIX: Scaler is fit on train_slice only (not all of train_orig)
            scaler = None
            if config.get("use_scaling", False):
                scaler = MinMaxScaler()
                # FIX: Uses unified get_feat_cols — same columns as Dataset and inference
                train_samples = [
                    pd.read_csv(f)[get_feat_cols(pd.read_csv(f).columns)].values
                    for f in train_slice
                ]
                scaler.fit(np.vstack(train_samples))

            # FIX: pos_weight computed from training slice only — no test-fold leak
            pos_weight = compute_pos_weight(train_slice, device)

            train_loader = DataLoader(
                ActivityDataset(get_with_mirrors(train_slice), config, scaler),
                batch_size=config["batch_size"], shuffle=True
            )
            val_loader = DataLoader(
                ActivityDataset(get_with_mirrors(val_slice), config, scaler),
                batch_size=config["batch_size"], shuffle=False
            )

            model     = ActivityGatekeeper(config).to(device)
            optimizer = optim.Adam(model.parameters(), lr=config["lr"])
            criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

            best_v_loss, patience_count = float('inf'), 0
            fold_ckpt = f"best_fold_{fold}_{config['note']}.pth"

            # Training loop
            for epoch in range(config["epochs"]):
                model.train()
                for x, y in train_loader:
                    optimizer.zero_grad()
                    criterion(model(x.to(device)), y.to(device)).backward()
                    optimizer.step()

                model.eval()
                v_loss = 0.0
                with torch.no_grad():
                    for x, y in val_loader:
                        v_loss += criterion(model(x.to(device)), y.to(device)).item()

                avg_v_loss = v_loss / len(val_loader)
                if avg_v_loss < best_v_loss:
                    best_v_loss    = avg_v_loss
                    patience_count = 0
                    torch.save(model.state_dict(), fold_ckpt)
                else:
                    patience_count += 1
                    if patience_count >= config["patience"]:
                        break

            # Boundary MAE evaluation on test fold
            model.load_state_dict(torch.load(fold_ckpt))
            model.eval()

            # FIX: fold_start_errs / fold_stop_errs are populated correctly so
            # the best-fold checkpoint save logic actually executes.
            fold_start_errs, fold_stop_errs = [], []

            for test_file in test_orig:
                df = pd.read_csv(test_file)

                actual_indices = df[df['activity_label'] == 1]['FrameNo'].values
                if len(actual_indices) == 0:
                    continue
                actual_start = actual_indices[0]
                actual_stop  = actual_indices[-1]

                # FIX: Unified get_feat_cols — same filter as scaler and Dataset
                feat_cols = get_feat_cols(df.columns)
                X_eval    = scaler.transform(df[feat_cols].values) if scaler else df[feat_cols].values.astype('float32')

                seq_len = config["seq_length"]
                preds   = [0] * (seq_len - 1)

                with torch.no_grad():
                    for i in range(len(df) - seq_len + 1):
                        window = torch.tensor(X_eval[i: i + seq_len]).float().unsqueeze(0).to(device)
                        preds.append(1 if torch.sigmoid(model(window)).item() >= 0.5 else 0)

                pred_indices = np.where(np.array(preds) == 1)[0]
                if len(pred_indices) > 0:
                    p_start   = df.iloc[pred_indices[0]]['FrameNo']
                    p_stop    = df.iloc[pred_indices[-1]]['FrameNo']
                    start_off = abs(p_start - actual_start)
                    stop_off  = abs(p_stop  - actual_stop)
                else:
                    # Penalty: model detected nothing
                    start_off = len(df)
                    stop_off  = len(df)

                total_off = start_off + stop_off

                fold_start_errs.append(start_off)
                fold_stop_errs.append(stop_off)
                # FIX: Appended to global lists ONLY here — the extend() calls
                # below are removed to prevent every error being counted twice.
                all_start_diffs.append(start_off)
                all_stop_diffs.append(stop_off)

                all_file_results.append({
                    "filename":  os.path.basename(test_file),
                    "start_off": start_off,
                    "stop_off":  stop_off,
                    "total_off": total_off,
                })

            # FIX: Removed all_start_diffs.extend() / all_stop_diffs.extend()
            # that caused double-counting. fold_* lists are now used only for
            # the fold-level MAE that governs which checkpoint to keep.
            if fold_start_errs:
                fold_mae = (np.mean(fold_start_errs) + np.mean(fold_stop_errs)) / 2

                if fold_mae < best_overall_mae:
                    best_overall_mae    = fold_mae
                    best_config_for_run = config.copy()
                    torch.save(model.state_dict(), ckpt_path)
                    if scaler is not None:
                        joblib.dump(scaler, scaler_path)

            if os.path.exists(fold_ckpt):
                os.remove(fold_ckpt)

        # ── End of fold loop ────────────────────────────────────────────

        report_df = pd.DataFrame(all_file_results)

        report_df.to_csv("full_file_performance.csv", index=False)
        mlflow.log_artifact("full_file_performance.csv")

        best_5 = report_df.nsmallest(5, 'total_off')
        best_5.to_csv("top_5_best_files.csv", index=False)
        mlflow.log_artifact("top_5_best_files.csv")

        worst_5 = report_df.nlargest(5, 'total_off')
        worst_5.to_csv("top_5_worst_files.csv", index=False)
        mlflow.log_artifact("top_5_worst_files.csv")

        mae_start  = np.mean(all_start_diffs)
        mae_stop   = np.mean(all_stop_diffs)
        std_start  = np.std(all_start_diffs)
        std_stop   = np.std(all_stop_diffs)
        total_std  = (std_start + std_stop) / 2   # FIX: was std_start + std_stop/2 (operator precedence bug)
        total_mae  = (mae_start + mae_stop) / 2

        mlflow.log_metric("MAE_Start_Frames",   mae_start)
        mlflow.log_metric("MAE_Stop_Frames",    mae_stop)
        mlflow.log_metric("MAE_Total_Average",  total_mae)

        print(f"\n--- Experiment: {config['note']} ---")
        print(f"Top 5 Worst Performers:\n{worst_5}")
        print(f"Top 5 Best Performers:\n{best_5}")
        print(f"\nFinal MAE Result -> Total: {total_mae:.2f} frames, std: {total_std:.2f}")
        print(f"Start: {mae_start:.2f} (±{std_start:.2f}) | Stop: {mae_stop:.2f} (±{std_stop:.2f})")

        # FIX: Guard against registering when no checkpoint was ever saved
        # (e.g. all test files had no positive labels, so fold_start_errs was
        # always empty and ckpt_path was never written).
        if utils.auto_check_challenger(run.info.run_id, metric_name="MAE_Total_Average") \
                and os.path.exists(ckpt_path):

            # FIX: model_to_log uses best_config_for_run (the config that
            # produced the saved checkpoint), not the current loop variable.
            model_to_log = ActivityGatekeeper(best_config_for_run).to(device)
            model_to_log.load_state_dict(torch.load(ckpt_path))

            mlflow.pytorch.log_model(
                model_to_log, "model", registered_model_name=project_name
            )

            latest_v = utils.client.get_latest_versions(project_name)[0].version
            utils.client.set_registered_model_alias(project_name, "dev", latest_v)
            print(f"Model ({config['note']}) registered to @dev")
        else:
            print("Performance did not beat @dev, or no valid checkpoint was saved.")




Initialized MLflow to track repo "SamuelFredricBerg/4dt907"

Repository SamuelFredricBerg/4dt907 initialized!

✅ Random seed set to: 42

--- Fold 1 ---

--- Fold 2 ---

--- Fold 3 ---

--- Fold 4 ---

--- Fold 5 ---

--- Fold 6 ---

--- Fold 7 ---

--- Fold 8 ---

--- Fold 9 ---

--- Fold 10 ---

--- Experiment: LSTM_Dense_Uni ---
Top 5 Worst Performers:
            filename  start_off  stop_off  total_off
48    A88_kinect.csv       84.0      46.0      130.0
7    A157_kinect.csv       85.0      14.0       99.0
109  A133_kinect.csv       38.0      48.0       86.0
57   A121_kinect.csv       73.0      10.0       83.0
171   A94_kinect.csv       47.0      24.0       71.0
Top 5 Best Performers:
            filename  start_off  stop_off  total_off
37   A125_kinect.csv        1.0       0.0        1.0
52    B14_kinect.csv        1.0       1.0        2.0
94    A34_kinect.csv        1.0       1.0        2.0
104   B16_kinect.csv        1.0       1.0        2.0
139   B12_kinect.csv        2.0       0.0        2.0

Final MAE Result -> Total: 11.12 frames, std: 13.18
Start: 11.84 (±14.64) | Stop: 10.39 (±11.7

2026/05/06 08:59:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 08:59:54 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'Start_Stop_Predictor_ModelV2' already exists. Creating a new version of this model...
2026/05/06 09:00:01 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Start_Stop_Predictor_ModelV2, version 6
Created version '6' of model 'Start_Stop_Predictor_ModelV2'.
/var/folders/9w/6516vw7x2gbc3llv8dbzm6vw0000gn/T/ipykernel_65691/1801594898.py:234: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: http

Model (LSTM_Dense_Uni) registered to @dev
🏃 View run A12 - new metrics at: https://dagshub.com/SamuelFredricBerg/4dt907.mlflow/#/experiments/0/runs/4380b764a18c49cba7162faabb619e28
🧪 View experiment at: https://dagshub.com/SamuelFredricBerg/4dt907.mlflow/#/experiments/0
✅ Random seed set to: 42

--- Fold 1 ---

--- Fold 2 ---

--- Fold 3 ---

--- Fold 4 ---

--- Fold 5 ---

--- Fold 6 ---

--- Fold 7 ---

--- Fold 8 ---

--- Fold 9 ---

--- Fold 10 ---

--- Experiment: GRU_Light_Rapid ---
Top 5 Worst Performers:
            filename  start_off  stop_off  total_off
48    A88_kinect.csv       77.0      41.0      118.0
90   A138_kinect.csv      103.0       7.0      110.0
19   A116_kinect.csv       40.0      64.0      104.0
109  A133_kinect.csv       41.0      59.0      100.0
134   A78_kinect.csv       58.0      25.0       83.0
Top 5 Best Performers:
            filename  start_off  stop_off  total_off
39    A19_kinect.csv        0.0       0.0        0.0
37   A125_kinect.csv        1.0    

In [ ]:
import torch
import pandas as pd
import numpy as np
import mlflow
import mlflow.pytorch
import joblib


def load_alias(alias, project_name, device):
    """Load model, config and scaler for a given registry alias."""
    client  = mlflow.MlflowClient()
    version = client.get_model_version_by_alias(project_name, alias)
    run     = client.get_run(version.run_id)

    type_map = {
        "seq_length": int, "hidden_size": int, "num_layers": int,
        "batch_size": int, "epochs": int, "patience": int, "input_size": int,
        "lr": float, "dropout": float,
        "use_scaling":    lambda v: v == "True",
        "bidirectional":  lambda v: v == "True",
    }
    config = {k: type_map[k](v) if k in type_map else v
              for k, v in run.data.params.items()}

    model = mlflow.pytorch.load_model(
        f"models:/{project_name}@{alias}", map_location=device
    ).to(device)
    model.eval()

    scaler = None
    if config.get("use_scaling"):
        scaler_path = client.download_artifacts(version.run_id, "scaler_best.joblib")
        scaler      = joblib.load(scaler_path)

    print(f"  [@{alias}] run={version.run_id[:8]}...  "
          f"note={config.get('note','?')}  "
          f"seq={config['seq_length']}  "
          f"scaler={'yes' if scaler else 'no'}")
    return model, config, scaler, version.run_id


def extract_exercise_bounds(file_path, config, model, device, scaler=None):
    """Return predicted (start_frame, stop_frame) for a single file."""
    df            = pd.read_csv(file_path)
    feat_cols     = get_feat_cols(df.columns)
    X_data        = df[feat_cols].values.astype('float32')
    if scaler is not None:
        X_data = scaler.transform(X_data)
    frame_numbers = df['FrameNo'].values

    seq_length  = config["seq_length"]
    predictions = []

    with torch.no_grad():
        for i in range(len(df) - seq_length + 1):
            window = torch.tensor(X_data[i: i + seq_length]).unsqueeze(0).to(device)
            prob   = torch.sigmoid(model(window)).item()
            predictions.append(1 if prob >= 0.5 else 0)

    full_preds  = [0] * (seq_length - 1) + predictions
    start_frame = stop_frame = None

    for i in range(1, len(full_preds)):
        if full_preds[i - 1] == 0 and full_preds[i] == 1 and start_frame is None:
            start_frame = frame_numbers[i]
        if full_preds[i - 1] == 1 and full_preds[i] == 0:
            stop_frame = frame_numbers[i - 1]

    # If the sequence ends on a 1, the last frame is the stop
    if stop_frame is None and full_preds[-1] == 1:
        stop_frame = frame_numbers[-1]

    # print(full_preds)

    return start_frame, stop_frame


# ── Setup ────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading models from MLflow registry...")
prod_model, prod_config, prod_scaler, prod_run_id = load_alias("prod", project_name, device)
dev_model,  dev_config,  dev_scaler,  dev_run_id  = load_alias("dev",  project_name, device)

# ── Ground truth ─────────────────────────────────────────────────────────────
test_csv    = "../../data/kinect_good_preprocessed_not_cut_A11_mediapipe/A57_kinect.csv"
labeled_csv = os.path.join("../../data/labeled_files", os.path.basename(test_csv))

df_labeled     = pd.read_csv(labeled_csv)
actual_indices = df_labeled[df_labeled['activity_label'] == 1]['FrameNo'].values

if len(actual_indices) == 0:
    raise ValueError(f"No ground-truth labels found in {labeled_csv}")

actual_start = int(actual_indices[0])
actual_stop  = int(actual_indices[-1])
actual_dur   = actual_stop - actual_start

# ── Inference ────────────────────────────────────────────────────────────────
prod_start, prod_stop = extract_exercise_bounds(test_csv, prod_config, prod_model, device, prod_scaler)
dev_start,  dev_stop  = extract_exercise_bounds(test_csv, dev_config,  dev_model,  device, dev_scaler)

# ── Report ───────────────────────────────────────────────────────────────────
def fmt(v): return f"{int(v):>10}" if v is not None else f"{'None':>10}"
def off(pred, actual): return abs(int(pred) - actual) if pred is not None else None
def fmt_off(v): return f"{v:>9} frames" if v is not None else f"{'N/A':>9}"

prod_start_off = off(prod_start, actual_start)
prod_stop_off  = off(prod_stop,  actual_stop)
dev_start_off  = off(dev_start,  actual_start)
dev_stop_off   = off(dev_stop,   actual_stop)

prod_total = (prod_start_off or 0) + (prod_stop_off or 0)
dev_total  = (dev_start_off  or 0) + (dev_stop_off  or 0)

W = 12
print(f"\nFile: {test_csv}\n")
print(f"{'':22s} {'Actual':>{W}} {'@prod':>{W}} {'off':>{W}} {'@dev':>{W}} {'off':>{W}}")
print("─" * 75)
print(f"{'Start frame':22s} {actual_start:{W}} {fmt(prod_start)} {fmt_off(prod_start_off)} {fmt(dev_start)} {fmt_off(dev_start_off)}")
print(f"{'Stop frame':22s} {actual_stop:{W}} {fmt(prod_stop)}  {fmt_off(prod_stop_off)} {fmt(dev_stop)}  {fmt_off(dev_stop_off)}")

prod_dur = (int(prod_stop) - int(prod_start)) if prod_start and prod_stop else None
dev_dur  = (int(dev_stop)  - int(dev_start))  if dev_start  and dev_stop  else None
print(f"{'Duration (frames)':22s} {actual_dur:{W}} {fmt(prod_dur)} {'':>{W+1}} {fmt(dev_dur)}")
print("─" * 75)
print(f"{'Total boundary error':22s} {'':>{W}} {'':>{W}} {fmt_off(prod_total)} {'':>{W}} {fmt_off(dev_total)}")
print()

winner = "@prod" if prod_total <= dev_total else "@dev"
print(f"→ {winner} is closer to ground truth on this file "
      f"(Δ = {abs(prod_total - dev_total)} frames)")

Loading models from MLflow registry...


  [@prod] run=432ba076...  note=GRU  seq=5  scaler=no


  [@dev] run=4380b764...  note=LSTM_Dense_Uni  seq=5  scaler=no
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0